# Telco Customer Churn — Phase 1: Data Cleaning

**Dataset:** IBM Telco Customer Churn (`WA_Fn-UseC_-Telco-Customer-Churn.csv`)  
**Objective:** Load the raw dataset, profile its structure and quality, resolve any issues, and export a clean file ready for EDA.

| Section | Description |
|---------|-------------|
| 0 | Setup & imports |
| 1 | Data loading |
| 2 | Initial profiling |
| 3 | Data quality assessment |
| 4 | Issue investigation — `TotalCharges` |
| 5 | Cleaning decisions |
| 6 | Data transformation |
| 7 | Post-cleaning validation |
| 8 | Final QA report |
| 9 | Before vs After |
| 10 | Export & verify |
| — | Phase 1 summary |

## 0 · Setup

### Analytical Grain

**One row = one customer**

The dataset is analyzed at the customer level. Each `customerID` represents one customer record. This grain is maintained throughout the cleaning process.

In [1]:
import pandas as pd

## 1 · Data Loading

Load the raw CSV from the project's `data/raw/` directory and take a first look at its shape and columns.

> **Repository layout assumed:**
> ```
> telco-churn-analysis/
> ├── data/
> │   ├── raw/
> │   │   └── WA_Fn-UseC_-Telco-Customer-Churn.csv
> │   └── processed/
> └── notebooks/
>     └── 01_data_cleaning.ipynb   ← run from here
> ```

In [2]:
DATA_PATH = "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = pd.read_csv(DATA_PATH)

print(f"Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")
df.head()

Rows: 7,043  |  Columns: 21


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
# Full column list
df.columns.tolist()

['customerID',
 'gender',
 'SeniorCitizen',
 'Partner',
 'Dependents',
 'tenure',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod',
 'MonthlyCharges',
 'TotalCharges',
 'Churn']

## 2 · Initial Profiling

Examine dtypes, null counts, and basic descriptive statistics before any cleaning.

In [4]:
# Data types and non-null counts
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [5]:
# Null values per column
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [6]:
# Whitespace-only cells (can masquerade as non-null)
(df == " ").sum()

customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

In [7]:
# Descriptive statistics for numeric columns
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


## 3 · Data Quality Assessment

Systematic checks across all quality dimensions before any cleaning is performed.

In [8]:
# Duplicate rows and customer IDs
print("Duplicate rows:       ", df.duplicated().sum())
print("Duplicate customerIDs:", df['customerID'].duplicated().sum())

Duplicate rows:        0
Duplicate customerIDs: 0


In [9]:
# Categorical value audit
cat_cols = [
    "gender", "Partner", "Dependents", "PhoneService", "MultipleLines",
    "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies", "Contract",
    "PaperlessBilling", "PaymentMethod", "Churn",
]

for col in cat_cols:
    print(f"\n{'─'*40}")
    print(f"  {col}")
    print(f"{'─'*40}")
    print(df[col].value_counts().to_string())


────────────────────────────────────────
  gender
────────────────────────────────────────
gender
Male      3555
Female    3488

────────────────────────────────────────
  Partner
────────────────────────────────────────
Partner
No     3641
Yes    3402

────────────────────────────────────────
  Dependents
────────────────────────────────────────
Dependents
No     4933
Yes    2110

────────────────────────────────────────
  PhoneService
────────────────────────────────────────
PhoneService
Yes    6361
No      682

────────────────────────────────────────
  MultipleLines
────────────────────────────────────────
MultipleLines
No                  3390
Yes                 2971
No phone service     682

────────────────────────────────────────
  InternetService
────────────────────────────────────────
InternetService
Fiber optic    3096
DSL            2421
No             1526

────────────────────────────────────────
  OnlineSecurity
────────────────────────────────────────
OnlineSecurity


In [10]:
# Numeric range audit — explicit constraint tests
numeric_validation = {
    "Invalid SeniorCitizen": int(
        (~df["SeniorCitizen"].isin([0, 1])).sum()
    ),
    "Invalid tenure (< 0 or > 72)": int(
        ((df["tenure"] < 0) | (df["tenure"] > 72)).sum()
    ),
    "Invalid MonthlyCharges (< 0)": int(
        (df["MonthlyCharges"] < 0).sum()
    ),
    "Invalid TotalCharges (< 0)": int(
        pd.to_numeric(df["TotalCharges"], errors="coerce").lt(0).sum()
    ),
}

pd.Series(numeric_validation)

Invalid SeniorCitizen           0
Invalid tenure (< 0 or > 72)    0
Invalid MonthlyCharges (< 0)    0
Invalid TotalCharges (< 0)      0
dtype: int64

In [11]:
# Cross-field logical consistency check
# Customers with no internet service should have 'No internet service'
# for all internet-dependent columns.
internet_dependent_cols = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies",
]

no_internet_mask = df["InternetService"] == "No"
logical_inconsistencies = 0

for col in internet_dependent_cols:
    invalid = df[no_internet_mask & (df[col] != "No internet service")]
    logical_inconsistencies += len(invalid)
    status = f"{len(invalid)} inconsistent records" if len(invalid) else "✓ clean"
    print(f"{col:<20}  {status}")

print(f"\nTotal logical inconsistencies: {logical_inconsistencies}")

OnlineSecurity        ✓ clean
OnlineBackup          ✓ clean
DeviceProtection      ✓ clean
TechSupport           ✓ clean
StreamingTV           ✓ clean
StreamingMovies       ✓ clean

Total logical inconsistencies: 0


## 4 · Issue Investigation — `TotalCharges`

`df.info()` shows `TotalCharges` stored as `object` despite representing a monetary measure. We first quantify blank values, then inspect the affected records.

In [12]:
# Preview the suspicious column
df[["tenure", "MonthlyCharges", "TotalCharges"]].head(10)

,tenure,MonthlyCharges,TotalCharges
0,1,29.85,29.85
1,34,56.95,1889.5
2,2,53.85,108.15
3,45,42.30,1840.75
4,2,70.70,151.65
5,8,99.65,820.5
6,22,89.10,1949.4
7,10,29.75,301.9
8,28,104.80,3046.05
9,62,56.15,3487.95


In [13]:
# Count whitespace-only values
print("Blank TotalCharges:  ", df["TotalCharges"].astype(str).str.strip().eq("").sum())
print("Blank MonthlyCharges:", df["MonthlyCharges"].astype(str).str.strip().eq("").sum())

Blank TotalCharges:   11
Blank MonthlyCharges: 0


In [14]:
# Inspect the 11 affected records
blank_mask = df["TotalCharges"].astype(str).str.strip().eq("")
df.loc[blank_mask, ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


### Interpretation

The 11 whitespace-only `TotalCharges` records were investigated individually.

All 11 affected records have `tenure = 0`. Since `TotalCharges` represents cumulative customer charges and these records have zero recorded tenure, the whitespace-only values are treated as **0** for this analysis.

This is an analytical assumption rather than a claim about the underlying billing system.

## 5 · Cleaning Decisions

| Issue | Evidence | Action |
|---|---|---|
| `TotalCharges` stored as `object` | Column represents a numeric monetary measure | Convert to `float64` |
| 11 whitespace-only `TotalCharges` values | All affected records have `tenure = 0` | Treat as 0 for this analysis |
| Duplicate rows | 0 identified | No action |
| Duplicate customer IDs | 0 identified | No action |
| Unexpected categorical values | None identified | No action |
| Invalid numeric values | None identified | No action |
| Logical inconsistencies | None identified | No action |

## 6 · Data Transformation

Apply the single correction identified above: replace whitespace-only `TotalCharges` entries with `0` and cast the column to `float64`.

In [15]:
# Replace whitespace-only strings with '0', then cast to numeric
df["TotalCharges"] = df["TotalCharges"].replace(r"^\s*$", "0", regex=True)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"])

print("New dtype:", df["TotalCharges"].dtype)
print("Remaining NaN:", df["TotalCharges"].isna().sum())

New dtype: float64
Remaining NaN: 0


## 7 · Post-Cleaning Validation

Confirm the fix is correct and that no unintended side-effects occurred.

In [16]:
# Verify tenure=0 records now hold 0.0 for TotalCharges
df.loc[
    df["tenure"] == 0,
    ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]
]

,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,0.0,No
753,3115-CZMZD,0,20.25,0.0,No
936,5709-LVOEQ,0,80.85,0.0,No
1082,4367-NUYAO,0,25.75,0.0,No
1340,1371-DWPAZ,0,56.05,0.0,No
3331,7644-OMVMY,0,19.85,0.0,No
3826,3213-VVOLG,0,25.35,0.0,No
4380,2520-SGTTA,0,20.00,0.0,No
5218,2923-ARZLG,0,19.70,0.0,No
6670,4075-WKNIU,0,73.35,0.0,No


In [17]:
# Confirm no remaining blank or null values
print("Blank TotalCharges:", df["TotalCharges"].astype(str).str.strip().eq("").sum())
print("Null  TotalCharges:", df["TotalCharges"].isna().sum())
print("Row count unchanged:", df.shape[0])

Blank TotalCharges: 0
Null  TotalCharges: 0
Row count unchanged: 7043


## 8 · Final QA Report

Consolidated pass/fail summary across all quality dimensions, including the logical consistency check from Section 3.

In [18]:
qa_results = {
    "Rows":                        df.shape[0],
    "Columns":                     df.shape[1],
    "Null values":                 int(df.isna().sum().sum()),
    "Duplicate rows":              int(df.duplicated().sum()),
    "Duplicate customer IDs":      int(df["customerID"].duplicated().sum()),
    "TotalCharges dtype":          str(df["TotalCharges"].dtype),
    "Blank TotalCharges":          int(df["TotalCharges"].astype(str).str.strip().eq("").sum()),
    "Logical inconsistencies":     logical_inconsistencies,
    "Invalid SeniorCitizen":       numeric_validation["Invalid SeniorCitizen"],
    "Invalid tenure":              numeric_validation["Invalid tenure (< 0 or > 72)"],
    "Invalid MonthlyCharges":      numeric_validation["Invalid MonthlyCharges (< 0)"],
    "Invalid TotalCharges":        numeric_validation["Invalid TotalCharges (< 0)"],
}

pd.Series(qa_results)

Rows                          7043
Columns                         21
Null values                      0
Duplicate rows                   0
Duplicate customer IDs           0
TotalCharges dtype         float64
Blank TotalCharges               0
Logical inconsistencies          0
Invalid SeniorCitizen            0
Invalid tenure                   0
Invalid MonthlyCharges           0
Invalid TotalCharges             0
dtype: object

## 9 · Before vs After

| Metric | Before Cleaning | After Cleaning |
|---|---:|---:|
| Rows | 7,043 | 7,043 |
| Columns | 21 | 21 |
| `TotalCharges` dtype | `object` | `float64` |
| Blank `TotalCharges` | 11 | 0 |
| Null values | 0 | 0 |
| Duplicate rows | 0 | 0 |
| Duplicate customer IDs | 0 | 0 |
| Logical inconsistencies | 0 | 0 |

## 10 · Export & Verify

Write the cleaned dataset to disk, then reload and confirm its integrity.

In [19]:
OUTPUT_PATH = "../data/processed/telco_cleaned.csv"

df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved → {OUTPUT_PATH}")

Saved → ../data/processed/telco_cleaned.csv


In [20]:
# Reload and verify
cleaned_df = pd.read_csv(OUTPUT_PATH)

assert cleaned_df.shape == (7043, 21), "Unexpected shape after export"
assert cleaned_df["TotalCharges"].dtype == float, "TotalCharges should be float"
assert cleaned_df.isna().sum().sum() == 0, "Unexpected nulls after export"
assert cleaned_df.duplicated().sum() == 0, "Unexpected duplicate rows after export"
assert cleaned_df["customerID"].duplicated().sum() == 0, "Unexpected duplicate customer IDs after export"
assert cleaned_df["TotalCharges"].astype(str).str.strip().eq("").sum() == 0, "Blank TotalCharges remain after export"

print("All assertions passed.")
print(f"Shape:                    {cleaned_df.shape}")
print(f"TotalCharges dtype:       {cleaned_df['TotalCharges'].dtype}")
print(f"Null values:              {cleaned_df.isna().sum().sum()}")
print(f"Duplicate rows:           {cleaned_df.duplicated().sum()}")
print(f"Duplicate customer IDs:   {cleaned_df['customerID'].duplicated().sum()}")
print(f"Blank TotalCharges:       {cleaned_df['TotalCharges'].astype(str).str.strip().eq('').sum()}")

All assertions passed.
Shape:                    (7043, 21)
TotalCharges dtype:       float64
Null values:              0


Duplicate rows:           0
Duplicate customer IDs:   0
Blank TotalCharges:       0


---

# Phase 1 — Data Cleaning Complete

## Summary

The Telco Customer Churn dataset contains **7,043 customer records and 21 columns** at customer-level grain.

The data-quality assessment covered:

- Missing values
- Blank/whitespace values
- Data types
- Categorical values
- Numeric validity
- Duplicate records
- Duplicate customer IDs
- Cross-field logical consistency

### Primary Issue Identified

`TotalCharges` was stored as `object` and contained **11 whitespace-only values**.

All 11 affected records had `tenure = 0`. These values were treated as 0 for this analysis and the column was converted to `float64`.

### Final Validation

All planned quality checks passed with no additional data corrections required.

The cleaned dataset was exported to:

`data/processed/telco_cleaned.csv`

**Phase 1 — Data Cleaning: COMPLETE ✅**